In [1]:
# Cell 1 - Imports
import sys
from pathlib import Path
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

sys.path.insert(0, str(Path("../scripts").resolve()))
from load_dataset import load_clauses

C:\Users\keros\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2 - Config
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1  # tiny test run - just proving the pipeline works

# Use the same 14 shortlisted categories as the baseline
CATEGORIES = [
    "License Grant", "Audit Rights", "Anti-Assignment", "Insurance",
    "Cap on Liability", "Non-Compete", "Revenue/Profit Sharing",
    "Minimum Commitment", "Exclusivity", "Termination for Convenience",
    "Post-Termination Services", "Non-Transferable License",
    "IP Ownership Assignment", "Change of Control"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Categories: {len(CATEGORIES)}")

Using device: cpu
Categories: 14


In [3]:
# Cell 3 - Load data using the same load_clauses function the baseline uses
print("Loading training data...")
train_clauses = load_clauses(split="train")
val_clauses = load_clauses(split="test")  # using test as val for this tiny run

# Filter to just the 14 shortlisted categories
train_clauses = train_clauses[train_clauses["category"].isin(CATEGORIES)]
val_clauses = val_clauses[val_clauses["category"].isin(CATEGORIES)]

print(f"Train rows: {len(train_clauses)}")
print(f"Val rows: {len(val_clauses)}")
print(f"Positive train examples: {train_clauses['has_annotation'].sum()}")

Loading training data...
Train rows: 5712
Val rows: 1428
Positive train examples: 2174


In [4]:
# Cell 4 - Build chunk-level labeled dataset
# A chunk is positive for a category if it overlaps an annotated span
# This matches exactly how the baseline labeled chunks

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_chunk_labels(clauses_df, chunk_size=512, overlap=64):
    """
    Convert clause-level rows into chunk-level rows with binary labels.
    Matches the chunking logic from chunking.ipynb and baseline_tfidf.py.
    """
    rows = []
    for _, row in clauses_df.iterrows():
        text = row["text"]
        category = row["category"]
        annotations = row["annotations"]
        contract_id = row["contract_id"]

        tokens = tokenizer.encode(text, add_special_tokens=False)
        start = 0
        chunk_index = 0

        while start < len(tokens):
            end = min(start + chunk_size, len(tokens))
            chunk_tokens = tokens[start:end]
            chunk_text = tokenizer.decode(chunk_tokens)

            # Approximate character positions for overlap check
            char_start = len(tokenizer.decode(tokens[:start]))
            char_end = len(tokenizer.decode(tokens[:end]))

            # A chunk is positive if it overlaps any annotation span
            label = 0
            for ann in annotations:
                if ann["start"] < char_end and ann["end"] > char_start:
                    label = 1
                    break

            rows.append({
                "contract_id": contract_id,
                "chunk_index": chunk_index,
                "chunk_text": chunk_text,
                "category": category,
                "label": label
            })

            chunk_index += 1
            start += chunk_size - overlap

    return pd.DataFrame(rows)

print("Building chunk-level labels for train (using small subset for test run)...")
# Use only first 50 contracts for this tiny training test
small_train = train_clauses[train_clauses["contract_id"].isin(
    train_clauses["contract_id"].unique()[:50]
)]
train_chunks = build_chunk_labels(small_train)
print(f"Train chunks: {len(train_chunks)}")
print(f"Positive chunks: {train_chunks['label'].sum()}")
print(f"Label distribution:\n{train_chunks['label'].value_counts()}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7556 > 512). Running this sequence through the model will result in indexing errors


Building chunk-level labels for train (using small subset for test run)...
Train chunks: 18438
Positive chunks: 550
Label distribution:
label
0    17888
1      550
Name: count, dtype: int64


In [5]:
# Cell 5 - PyTorch Dataset class
class CUADChunkDataset(Dataset):
    def __init__(self, chunks_df, tokenizer, max_length=512):
        self.chunks = chunks_df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        row = self.chunks.iloc[idx]
        encoding = self.tokenizer(
            row["chunk_text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

train_dataset = CUADChunkDataset(train_chunks, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Dataset size: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# Inspect one batch to verify shapes
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_ids: {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  labels: {batch['label'].shape}")

Dataset size: 18438
Batches per epoch: 2305

Batch shapes:
  input_ids: torch.Size([8, 512])
  attention_mask: torch.Size([8, 512])
  labels: torch.Size([8])


In [6]:
# Cell 6 - Load model
print(f"Loading {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # binary: positive or negative for each category
)
model = model.to(device)
print("Model loaded successfully")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading bert-base-uncased...


C:\Users\keros\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\keros\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:

Model loaded successfully
Parameters: 109,483,778


In [7]:
# Cell 7 - Tiny training test (just a few batches to confirm everything runs)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print("Running tiny training test (5 batches only)...")
model.train()
losses = []

for batch_num, batch in enumerate(train_loader):
    if batch_num >= 5:  # just 5 batches to prove it works
        break

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f"Batch {batch_num + 1}/5 | Loss: {loss.item():.4f}")

print(f"\nAverage loss over 5 batches: {sum(losses)/len(losses):.4f}")
print("Tiny training test complete - pipeline is working!")

Running tiny training test (5 batches only)...
Batch 1/5 | Loss: 1.0710
Batch 2/5 | Loss: 0.4879
Batch 3/5 | Loss: 0.3740
Batch 4/5 | Loss: 0.3693
Batch 5/5 | Loss: 0.2921

Average loss over 5 batches: 0.5189
Tiny training test complete - pipeline is working!


In [8]:
# Cell 8 - Quick inference check on one example
print("Running inference on one example...")
model.eval()

sample = train_chunks[train_chunks["label"] == 1].iloc[0]
print(f"Category: {sample['category']}")
print(f"Label: {sample['label']}")
print(f"Text preview: {sample['chunk_text'][:200]}")

encoding = tokenizer(
    sample["chunk_text"],
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(
        input_ids=encoding["input_ids"].to(device),
        attention_mask=encoding["attention_mask"].to(device)
    )
    probs = torch.softmax(outputs.logits, dim=-1)
    pred = torch.argmax(probs, dim=-1).item()

print(f"\nPredicted class: {pred}")
print(f"Probabilities: negative={probs[0][0]:.3f}, positive={probs[0][1]:.3f}")
print("Inference working correctly!")

Running inference on one example...
Category: Exclusivity
Label: 1
Text preview: exhibit 10. 6 distributor agreement this distributor agreement ( the " agreement " ) is made by and between electric city corp., a delaware corporation ( " company " ) and electric city of illinois ll

Predicted class: 0
Probabilities: negative=0.819, positive=0.181
Inference working correctly!
